# 01 — Preprocessing the rate-sweep dataset (catalog-fixed / best data)

This is the **corrected swept** ndnSIM dataset. The earlier sweep had a fatal flaw —
the legitimate background traffic died ~30 s into every run, so the attack window was
attacker-only. That is now fixed (`NumberOfContents = 300`): the legitimate consumers
emit a sustained ~4.4 interests/s each for the entire 600 s run. Every attack scenario
was regenerated across a range of attacker rates and replicated over seeds (low rates
10–30 have 5 seeds; high anchors 50/100 have 1 seed each — 177 runs total).

Naming convention:
- attack:  `{topo}-{attack}-r{rate}-run{seed}-{tracer}.{ext}`
- normal:  `{topo}-normal-run{seed}-{tracer}.{ext}`

The goal of this notebook is the same as the original `01_preprocessing` —
turn raw tracers into per-node-per-second feature rows — but it carries two extra
dimensions through the whole pipeline: **attacker rate** and **seed (run)**.

Output: `processed/full_sweep.csv` (one row per node-second-scenario-rate-run) plus a
compact `processed/sweep_node_summary.csv` (pre-attack vs attack-window means per node).

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "ndnsim-research-main" / "ndn-research" / "results"
SAVE_DIR = PROJECT_ROOT / "processed"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TOPOLOGIES = ["tree", "dfn", "dumbbell"]
ATTACKS    = ["ifa", "cp"]

# Node IDs derived from the scenario .cpp creation order (verified against the traces):
#   tree / dfn : producers 0-1, routers 2-5, consumers 6-11  (c1=6 ... c6=11)
#   dumbbell   : consumers 0-2, producers 3-4, routers 5-9   (c1=0,c2=1,c3=2)
ATTACKER_NODES = {
    ("tree",     "ifa"): {6},
    ("dfn",      "ifa"): {6},
    ("dumbbell", "ifa"): {0},
    ("tree",     "cp"):  {6, 11},
    ("dfn",      "cp"):  {6, 11},
    ("dumbbell", "cp"):  {0, 2},
}

print("DATA_DIR:", DATA_DIR)
print("exists  :", DATA_DIR.exists())


DATA_DIR: /Users/ankitpokhrel/Desktop/minor_project_refactored/NDNsim_best_data/ndnsim-research-main/ndn-research/results
exists  : True


In [2]:
# Enumerate every run from the rate-trace files (one per run).
pat_atk = re.compile(r"^(?P<topo>\w+)-(?P<attack>ifa|cp)-r(?P<rate>\d+)-run(?P<run>\d+)-rate-trace\.txt$")
pat_nrm = re.compile(r"^(?P<topo>\w+)-normal-run(?P<run>\d+)-rate-trace\.txt$")

runs = []
for p in sorted(DATA_DIR.glob("*-rate-trace.txt")):
    m = pat_atk.match(p.name)
    if m:
        d = m.groupdict()
        runs.append(dict(topo=d["topo"], scenario=d["attack"], rate=int(d["rate"]), run=int(d["run"])))
        continue
    m = pat_nrm.match(p.name)
    if m:
        d = m.groupdict()
        runs.append(dict(topo=d["topo"], scenario="normal", rate=np.nan, run=int(d["run"])))

runs = pd.DataFrame(runs)
print("Total runs:", len(runs))
print("\nRuns per (topo, scenario):")
print(runs.groupby(["topo","scenario"]).size())
print("\nRates present (attacks):", sorted(runs.loc[runs.scenario!="normal","rate"].dropna().unique().astype(int)))
print("Seeds present:", sorted(runs.run.unique()))


Total runs: 177

Runs per (topo, scenario):
topo      scenario
dfn       cp          27
          ifa         27
          normal       5
dumbbell  cp          27
          ifa         27
          normal       5
tree      cp          27
          ifa         27
          normal       5
dtype: int64

Rates present (attacks): [np.int64(10), np.int64(12), np.int64(15), np.int64(20), np.int64(30), np.int64(50), np.int64(100)]
Seeds present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [3]:
def _stem(topo, scenario, rate, run):
    if scenario == "normal":
        return f"{topo}-normal-run{run}"
    return f"{topo}-{scenario}-r{int(rate)}-run{run}"

def load_rate_trace(stem):
    rt = pd.read_csv(DATA_DIR / f"{stem}-rate-trace.txt", sep="\t", comment="#",
                     usecols=["Time","Node","Type","Packets"])
    KEEP = {"InInterests","OutInterests","InData","InNacks",
            "InSatisfiedInterests","InTimedOutInterests"}
    rt = rt[rt["Type"].isin(KEEP)]
    out = (rt.groupby(["Time","Node","Type"])["Packets"].sum()
             .unstack("Type").fillna(0.0).reset_index())
    out.columns.name = None
    for c in KEEP:
        if c not in out.columns:
            out[c] = 0.0
    return out

def load_cs_trace(stem):
    cs = pd.read_csv(DATA_DIR / f"{stem}-cs-trace.txt", sep="\t", comment="#")
    cs.columns = cs.columns.str.strip()
    cs = cs[["Time","Node","Type","Packets"]]
    out = (cs.groupby(["Time","Node","Type"])["Packets"].sum()
             .unstack("Type").fillna(0.0).reset_index())
    out.columns.name = None
    return out

def load_app_delays(stem):
    ad = pd.read_csv(DATA_DIR / f"{stem}-app-delays.txt", sep="\t", comment="#")
    ad.columns = ad.columns.str.strip()
    ad = ad[ad["Type"] == "FullDelay"].copy()
    if ad.empty:
        return pd.DataFrame(columns=["Time","Node","delay_mean","retx_mean","hop_mean","n_satisfied"])
    ad["Time"] = ad["Time"].apply(np.floor).astype(int).clip(lower=1)
    return (ad.groupby(["Time","Node"])
              .agg(delay_mean=("DelayS","mean"),
                   retx_mean=("RetxCount","mean"),
                   hop_mean=("HopCount","mean"),
                   n_satisfied=("DelayS","count"))
              .reset_index())

def load_ground_truth(stem):
    gt = pd.read_csv(DATA_DIR / f"{stem}-ground-truth.csv")
    gt.columns = gt.columns.str.strip().str.lower()
    gt = gt.rename(columns={"time":"Time"})
    gt["Time"] = gt["Time"] + 1          # GT 0-indexed, traces 1-indexed
    return gt[(gt["Time"]>=1)&(gt["Time"]<=599)]

# smoke test
s = _stem("tree","ifa",100,1)
print(load_rate_trace(s).shape, load_cs_trace(s).shape, load_app_delays(s).shape, load_ground_truth(s).shape)


(7188, 8) (7188, 4) (3594, 6) (599, 2)


In [4]:
# Consumer nodes = nodes that ever report an application delay in the normal runs.
CONSUMER_NODES = {}
for topo in TOPOLOGIES:
    ad = load_app_delays(_stem(topo,"normal",np.nan,1))
    CONSUMER_NODES[topo] = set(ad["Node"].unique())
print("Consumers per topology:", {k:sorted(v) for k,v in CONSUMER_NODES.items()})
print("Attacker nodes        :", {f"{k[0]}-{k[1]}":sorted(v) for k,v in ATTACKER_NODES.items()})


Consumers per topology: {'tree': [np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)], 'dfn': [np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)], 'dumbbell': [np.int64(0), np.int64(1), np.int64(2)]}
Attacker nodes        : {'tree-ifa': [6], 'dfn-ifa': [6], 'dumbbell-ifa': [0], 'tree-cp': [6, 11], 'dfn-cp': [6, 11], 'dumbbell-cp': [0, 2]}


In [5]:
def engineer(topo, scenario, rate, run):
    stem = _stem(topo, scenario, rate, run)
    rt = load_rate_trace(stem)
    cs = load_cs_trace(stem)
    ad = load_app_delays(stem)
    gt = load_ground_truth(stem)

    df = rt.merge(cs, on=["Time","Node"], how="left").merge(ad, on=["Time","Node"], how="left")
    for c in ["delay_mean","retx_mean","hop_mean","n_satisfied"]:
        if c not in df.columns: df[c] = 0.0
    df[["delay_mean","retx_mean","hop_mean","n_satisfied"]] = df[["delay_mean","retx_mean","hop_mean","n_satisfied"]].fillna(0.0)

    for c in ["CacheHits","CacheMisses"]:
        if c not in df.columns: df[c] = 0.0
    total_cs = df["CacheHits"] + df["CacheMisses"]
    df["cache_hit_ratio"]   = np.where(total_cs>0, df["CacheHits"]/total_cs, 0.0)
    df["satisfaction_ratio"]= np.where(df["InInterests"]>0, df["InSatisfiedInterests"]/df["InInterests"], 0.0).clip(0,1)
    df["timeout_ratio"]     = np.where(df["InInterests"]>0, df["InTimedOutInterests"]/df["InInterests"], 0.0).clip(0,1)
    df["nack_ratio"]        = np.where(df["InInterests"]>0, df["InNacks"]/df["InInterests"], 0.0).clip(0,1)
    df["interest_amp"]      = np.where(df["InInterests"]>0, df["OutInterests"]/df["InInterests"], 1.0)
    df["data_ratio"]        = np.where(df["OutInterests"]>0, df["InData"]/df["OutInterests"], 0.0).clip(0,1)

    attackers = ATTACKER_NODES.get((topo,scenario), set())
    def role(n):
        if n in attackers: return "attacker"
        if n in CONSUMER_NODES[topo]: return "benign_consumer"
        return "router"
    df["role"] = df["Node"].map(role)

    df = df.merge(gt[["Time","label"]], on="Time", how="left")
    df["label"] = df["label"].fillna("normal")

    df["topology"] = topo
    df["scenario"] = scenario
    df["rate"]     = rate
    df["run"]      = run
    return df

t = engineer("tree","ifa",10,1)
print("sample shape:", t.shape)
print(t[["Time","Node","role","InInterests","OutInterests","label"]].query("Node==6").head(3).to_string())


sample shape: (7188, 26)
    Time  Node      role  InInterests  OutInterests   label
6      1     6  attacker        7.200         7.200  normal
18     2     6  attacker        9.440         9.440  normal
30     3     6  attacker        9.888         7.488  normal


In [6]:
import time
parts = []
t0 = time.time()
for i, r in runs.iterrows():
    parts.append(engineer(r.topo, r.scenario, r.rate, r.run))
    if (i+1) % 45 == 0:
        print(f"  {i+1}/{len(runs)} runs   ({time.time()-t0:.0f}s)")
full = pd.concat(parts, ignore_index=True)
print("\nfull_sweep shape:", full.shape)
print("by scenario:\n", full.groupby("scenario").size())


  45/177 runs   (8s)


  90/177 runs   (14s)


  135/177 runs   (20s)



full_sweep shape: (1196377, 26)
by scenario:
 scenario
cp        544665
ifa       549882
normal    101830
dtype: int64


In [7]:
full.to_csv(SAVE_DIR / "full_sweep.csv", index=False)
print("saved full_sweep.csv", full.shape)

# Compact summary: per (topo,scenario,rate,run,Node,role) pre-attack vs attack-window means
ATTACK_START = 301
key = ["topology","scenario","rate","run","Node","role"]
feat = ["InInterests","OutInterests","InData","cache_hit_ratio","satisfaction_ratio",
        "timeout_ratio","nack_ratio","interest_amp","data_ratio"]
pre  = (full[full.Time<=300].groupby(key)[feat].mean().add_suffix("_pre"))
atk  = (full[full.Time>=ATTACK_START].groupby(key)[feat].mean().add_suffix("_atk"))
summary = pre.join(atk, how="outer").reset_index()
summary.to_csv(SAVE_DIR / "sweep_node_summary.csv", index=False)
print("saved sweep_node_summary.csv", summary.shape)


saved full_sweep.csv (1196377, 26)


saved sweep_node_summary.csv (1836, 24)


In [8]:
for f in ["full_sweep.csv","sweep_node_summary.csv"]:
    p = SAVE_DIR / f
    print(f"{f:<28} {p.stat().st_size/1e6:8.1f} MB")
print("\nDone.")


full_sweep.csv                  205.7 MB
sweep_node_summary.csv            0.5 MB

Done.
